# Overcooked Zero-Shot League Colab
生成マップをまたいで学習し、未知マップ評価に備えます。

In [ ]:
from pathlib import Path
import math, json
from marl_course.algos.coop_bandit import CoopStrategyBanditPolicy
from marl_course.algos.logging import MetricLogger
from marl_course.envs.coop_kitchen import CoopKitchenEnv, builtin_layouts, generate_layout
from marl_course.evaluation.coop import make_team_obs


In [ ]:
config = json.loads(Path('configs/train_coop_zero_shot.json').read_text())
config.update({'episodes': 50, 'out': 'outputs/colab_coop_zero_shot', 'wandb': False})
out = Path(config['out'])
policy = CoopStrategyBanditPolicy(seed=config['seed'])
logger = MetricLogger(out, use_wandb=config['wandb'], project=config['wandb_project'], run_name='colab-coop-zero-shot', config=config)


In [ ]:
best_score = 0
base_layouts = list(builtin_layouts().values())
for ep in range(config['episodes']):
    epsilon = config['epsilon_end'] + (config['epsilon_start'] - config['epsilon_end']) * math.exp(-3.0 * ep / config['episodes'])
    if ep < len(base_layouts):
        layout = base_layouts[ep]
        bucket = 'seen'
    else:
        bucket = config['families'][ep % len(config['families'])]
        layout = generate_layout(200 + ep, family=bucket)
    env = CoopKitchenEnv(layout=layout)
    obs, _ = env.reset(seed=200 + ep)
    policy.reset_episode({'layout': layout.name, 'bucket': bucket}, epsilon=epsilon)
    done = False
    while not done:
        team_obs = make_team_obs(obs)
        actions = policy.act(team_obs, team_obs['action_mask'], deterministic=False)
        result = env.step({f'agent_{i}': actions[i] for i in range(4)})
        obs = result.observations
        done = any(result.truncations.values())
    target = env.score - 0.01 * env.collisions
    loss = policy.update(target, alpha=config['alpha'])
    best_score = max(best_score, env.score)
    logger.log({'episode': ep, 'score': env.score, 'soups': env.delivered_soups, 'best_score': best_score, 'epsilon': epsilon, 'layout_bucket': bucket, 'strategy': policy.current_strategy, 'loss': loss, 'collisions': env.collisions}, step=ep)
logger.close()


In [ ]:
out.mkdir(parents=True, exist_ok=True)
policy.save(out / 'policy.pt')
(out / 'metadata.json').write_text(json.dumps({'student_id': 'colab_coop_zero_shot', 'env_id': 'coop_kitchen_zero_shot_v1'}, indent=2), encoding='utf-8')
(out / 'policy.py').write_text("from pathlib import Path\nfrom marl_course.algos.coop_bandit import CoopStrategyBanditPolicy\n\ndef load_policy(model_path, device='cpu'):\n    return CoopStrategyBanditPolicy.load(Path(model_path))\n", encoding='utf-8')


In [ ]:
import pandas as pd
pd.read_csv(out / 'metrics.csv').plot(x='episode', y=['score', 'best_score', 'epsilon'], subplots=True, figsize=(8, 6));
